# ⚖️ nnU-Net vs Swin UNETR — the comparison study

**📎 Required Kaggle Datasets (attach BOTH):**
1. Output of **notebook 02** → contains `per_case_metrics.csv` / `summary.json` (nnU-Net arm)
2. Output of **notebook 03** → contains `per_case_metrics.csv` / `summary.json` (Swin arm)

This notebook builds:
- `comparison_table.md/.csv` — Dice & HD95 per label per model (mean ± std, median)
- `boxplot_per_case_dice.png` — per-case distributions (means hide variance — plots don't)
- a short **findings template** you should fill in by hand for your repo's writeup

In [ ]:
!git clone --depth 1 https://github.com/naveed1011/nnunet-medseg-benchmark.git /kaggle/working/repo
import os, glob, sys, json
sys.path.insert(0, "/kaggle/working/repo")
os.chdir("/kaggle/working/repo")

In [ ]:
# locate the two arms' results among attached inputs. Both write
# per_case_metrics.csv, and "swinunetr" contains "nnunet" — so match the
# immediate parent folder name exactly instead of doing a substring search.
import os, glob
csvs = glob.glob("/kaggle/input/**/per_case_metrics.csv", recursive=True) + \
       glob.glob("/kaggle/working/**/per_case_metrics.csv", recursive=True)
print("found result CSVs:")
for c in csvs: print("  ", c)

def pick(name):
    hits = [os.path.dirname(c) for c in csvs
            if os.path.basename(os.path.dirname(c)).lower() == name.lower()]
    assert hits, (f"No results folder named '{name}' — attach both notebook outputs "
                  f"(found: {[os.path.dirname(c) for c in csvs]})")
    return hits[0]

NNUNET_DIR = pick("brats_nnunet")
SWIN_DIR   = pick("brats_swinunetr")
assert NNUNET_DIR != SWIN_DIR, "both arms resolved to the same folder — fix the paths below"
print("nnU-Net arm:", NNUNET_DIR)
print("Swin arm:   ", SWIN_DIR)

In [ ]:
%%time
# If the two CSVs are ambiguous by name, point the paths manually:
# NNUNET_DIR = "/kaggle/input/<nb02-output>/results/brats_nnunet"
# SWIN_DIR   = "/kaggle/input/<nb03-output>/results/brats_swinunetr"
!python -m src.compare --arm nnunet={NNUNET_DIR} --arm swinunetr={SWIN_DIR} --out /kaggle/working/comparison --task "BraTS2021 subset (NCR/ED/ET)"
print(open("/kaggle/working/comparison/comparison_table.md").read())

In [ ]:
from IPython.display import Image, display
display(Image("/kaggle/working/comparison/figures/boxplot_per_case_dice.png"))
display(Image("/kaggle/working/comparison/figures/bar_dice_by_label.png"))

## 5 · Package the comparison for the repo

`src.compare` writes `RESULTS.md` next to the figures, so the repo's results page and the
Kaggle output cannot drift apart. Copy this folder into `docs/results/brats/` when you push.

In [ ]:
import shutil
out = "/kaggle/working/outputs_comparison"
shutil.rmtree(out, ignore_errors=True)
shutil.copytree("/kaggle/working/comparison", out)
shutil.copy2("/kaggle/working/split/brats_subset_split.json", out)
print("📦 outputs_comparison:", os.listdir(out))
print(open(f"{out}/RESULTS.md").read()[:1500])

## 📝 Findings template (fill this in — this is the research contribution!)

> **Setup.** Both models trained on the same {N_train}-case BraTS 2021 subset (80/20 fixed
> split), 100-epoch budget, evaluated on the same {N_val} held-out volumes with Dice + HD95-mm.
> nnU-Net self-configured: patch {…}, batch {…}, spacing {…} (read from `nnUNetPlans.json`).
> Swin UNETR: 96³ patches, batch 1, matched preprocessing.
>
> **Observations.** (examples of what to look for)
> - Which model wins per structure? ET is small → HD95 punishes misses; ED is diffuse → Dice differs less.
> - Variance: does one model have heavier tails (a few catastrophic cases)?
> - Efficiency: params, GPU memory, inference s/volume (see `model_stats.json`).
>
> **Interpretation.** Why might a self-configured CNN beat (or lose to) a pretrained Transformer
> at this data scale? (small-data regime, augmentation strength, inductive bias, pretraining domain gap)
>
> **Limitations.** subset size, reduced epochs, budget-matched ≠ tuning-matched, single fold,
> no ensembling/TTA for saliency runs, no significance test at n={N_val}.